In [ ]:
"""
CCTV Anomaly Detection and Report Generation
-------------------------------------------
This notebook provides an end-to-end solution for analyzing CCTV footage,
detecting anomalies, and generating professional PDF reports.
"""

# Install required packages
!pip install -q torch transformers opencv-python pillow reportlab spacy matplotlib tqdm

# Download language model for grammar correction
!python -m spacy download en_core_web_sm

# Import necessary libraries
import os
import cv2
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
from datetime import datetime, timedelta
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import spacy
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from transformers import BlipProcessor, BlipForConditionalGeneration
from google.colab import files
import re
import shutil
import json

# Define the CCTVAnalyzer class
class CCTVAnalyzer:
    def __init__(self):
        print("Initializing models...")

        # Initialize object detection model (DETR)
        self.obj_processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
        self.obj_model = AutoModelForObjectDetection.from_pretrained("facebook/detr-resnet-50")

        # Initialize image captioning model (BLIP)
        self.caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
        self.caption_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

        # Initialize NLP model for grammar correction
        self.nlp = spacy.load("en_core_web_sm")

        # Define keyword sets for different anomaly types
        self.anomaly_keywords = {
            "violence": ["fighting", "punch", "hit", "attack", "struggle", "assault", "violent", "weapon", "knife", "gun", "beat"],
            "theft": ["steal", "robbery", "theft", "shoplifting", "grab", "snatch", "taking", "burglar"],
            "suspicious": ["lurking", "hiding", "suspicious", "strange", "unusual", "odd behavior", "mask", "conceal"],
            "emergency": ["fall", "collapse", "accident", "injured", "hurt", "emergency", "medical", "fire", "smoke"],
            "vandalism": ["damage", "break", "destroy", "graffiti", "vandal", "smash", "wreck", "trash"]
        }

        print("Models loaded successfully!")

    def extract_frames(self, video_path, output_dir, frame_interval=30):
        """Extract frames at specified intervals from a video"""
        os.makedirs(output_dir, exist_ok=True)

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = frame_count / fps

        # Extract video metadata
        metadata = {
            "fps": fps,
            "frame_count": frame_count,
            "duration": duration,
            "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        }

        # Calculate timestamps and frames to extract
        frames_to_extract = list(range(0, frame_count, frame_interval))
        frames_data = []

        print(f"Extracting {len(frames_to_extract)} frames from video...")

        for i, frame_idx in enumerate(tqdm(frames_to_extract)):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()

            if not ret:
                continue

            timestamp = frame_idx / fps
            formatted_time = str(timedelta(seconds=int(timestamp)))

            # Save the frame
            frame_path = os.path.join(output_dir, f"frame_{i:04d}.jpg")
            cv2.imwrite(frame_path, frame)

            frames_data.append({
                "index": i,
                "original_frame_idx": frame_idx,
                "timestamp": timestamp,
                "formatted_time": formatted_time,
                "path": frame_path
            })

        cap.release()

        return frames_data, metadata

    def detect_objects(self, image_path):
        """Detect objects in an image using DETR"""
        image = Image.open(image_path)
        inputs = self.obj_processor(images=image, return_tensors="pt")
        outputs = self.obj_model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]])
        results = self.obj_processor.post_process_object_detection(
            outputs, threshold=0.7, target_sizes=target_sizes
        )[0]

        detections = []
        for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
            detections.append({
                "label": self.obj_model.config.id2label[label.item()],
                "score": score.item(),
                "box": box.tolist()
            })

        return detections

    def generate_caption(self, image_path):
        """Generate a caption for an image using BLIP"""
        image = Image.open(image_path).convert('RGB')
        inputs = self.caption_processor(image, return_tensors="pt")

        # Generate caption
        out = self.caption_model.generate(**inputs, max_length=50)
        caption = self.caption_processor.decode(out[0], skip_special_tokens=True)

        # Improve grammar and formatting
        doc = self.nlp(caption)
        improved_caption = re.sub(r'\s+', ' ', caption).strip()

        # Ensure caption begins with a capital letter and ends with a period
        improved_caption = improved_caption[0].upper() + improved_caption[1:]
        if not improved_caption.endswith('.'):
            improved_caption += '.'

        return improved_caption

    def check_for_anomalies(self, caption, detections):
        """Check if the caption contains keywords indicating anomalies"""
        caption_lower = caption.lower()

        # Check each anomaly type
        for anomaly_type, keywords in self.anomaly_keywords.items():
            for keyword in keywords:
                if keyword in caption_lower:
                    return {"type": anomaly_type, "keyword": keyword}

        # Check for people-related anomalies based on object detection
        person_count = sum(1 for det in detections if det["label"] == "person")

        if person_count >= 5:
            return {"type": "crowd", "keyword": "multiple people"}

        return None

    def analyze_frame(self, frame_data):
        """Analyze a single frame for objects and anomalies"""
        image_path = frame_data["path"]

        # Detect objects
        detections = self.detect_objects(image_path)

        # Generate caption
        caption = self.generate_caption(image_path)

        # Check for anomalies
        anomaly = self.check_for_anomalies(caption, detections)

        return {
            **frame_data,
            "caption": caption,
            "detections": detections,
            "anomaly": anomaly
        }

    def analyze_video(self, video_path, output_dir, frame_interval=30):
        """Analyze an entire video and generate a report"""
        print("Starting video analysis...")

        # Create output directory
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        frames_dir = os.path.join(output_dir, "frames")
        analyzed_dir = os.path.join(output_dir, "analyzed")
        os.makedirs(frames_dir, exist_ok=True)
        os.makedirs(analyzed_dir, exist_ok=True)

        # Extract frames
        frames_data, video_metadata = self.extract_frames(video_path, frames_dir, frame_interval)

        # Analyze each frame
        print(f"Analyzing {len(frames_data)} frames...")
        analyzed_frames = []
        anomalies = []

        for frame_data in tqdm(frames_data):
            analysis = self.analyze_frame(frame_data)
            analyzed_frames.append(analysis)

            if analysis["anomaly"]:
                anomalies.append({
                    "frame_index": analysis["index"],
                    "timestamp": analysis["formatted_time"],
                    "caption": analysis["caption"],
                    "type": analysis["anomaly"]["type"],
                    "keyword": analysis["anomaly"]["keyword"],
                    "path": analysis["path"]
                })

        # Generate summary
        summary = self.generate_summary(analyzed_frames, anomalies, video_metadata)

        # Generate HTML report
        html_report_path = os.path.join(output_dir, "report.html")
        self.generate_html_report(analyzed_frames, anomalies, summary, video_metadata, html_report_path)

        # Generate PDF report
        pdf_report_path = os.path.join(output_dir, "report.pdf")
        self.generate_pdf_report(analyzed_frames, anomalies, summary, video_metadata, pdf_report_path)

        # Save analysis JSON
        json_path = os.path.join(output_dir, "analysis.json")
        with open(json_path, 'w') as f:
            json.dump({
                "metadata": video_metadata,
                "summary": summary,
                "anomalies": anomalies,
                "frames": [
                    {k: v for k, v in frame.items() if k != 'detections'}
                    for frame in analyzed_frames
                ]
            }, f, indent=2)

        return {
            "summary": summary,
            "anomalies": anomalies,
            "html_report": html_report_path,
            "pdf_report": pdf_report_path,
            "json_analysis": json_path
        }

    def generate_summary(self, analyzed_frames, anomalies, metadata):
        """Generate a summary of the video analysis"""
        total_frames = len(analyzed_frames)
        anomaly_count = len(anomalies)
        duration_mins = metadata["duration"] / 60

        if anomaly_count == 0:
            return f"No anomalies detected during {duration_mins:.1f} minutes of footage. Normal activity observed."

        # Group anomalies by type
        anomaly_types = {}
        for anomaly in anomalies:
            if anomaly["type"] not in anomaly_types:
                anomaly_types[anomaly["type"]] = 0
            anomaly_types[anomaly["type"]] += 1

        anomaly_list = ", ".join([f"{count} instances of {atype}" for atype, count in anomaly_types.items()])

        return f"Detected {anomaly_count} anomalies ({anomaly_list}) during {duration_mins:.1f} minutes of footage. Further review recommended."

    def generate_html_report(self, analyzed_frames, anomalies, summary, metadata, output_path):
        """Generate an HTML report of the analysis"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        duration = str(timedelta(seconds=int(metadata["duration"])))

        html = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>CCTV Analysis Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                .header {{ background-color: #4285f4; color: white; padding: 20px; margin-bottom: 20px; }}
                .section {{ margin-bottom: 30px; }}
                .anomaly {{ background-color: #ffdddd; padding: 15px; margin-bottom: 15px; border-radius: 5px; }}
                .frames {{ display: flex; flex-wrap: wrap; }}
                .frame {{ margin: 10px; text-align: center; }}
                img {{ max-width: 320px; max-height: 240px; }}
                table {{ border-collapse: collapse; width: 100%; }}
                th, td {{ padding: 10px; text-align: left; border-bottom: 1px solid #ddd; }}
                th {{ background-color: #f2f2f2; }}
            </style>
        </head>
        <body>
            <div class="header">
                <h1>CCTV Footage Analysis Report</h1>
                <p>Generated on: {timestamp}</p>
            </div>

            <div class="section">
                <h2>Video Summary</h2>
                <table>
                    <tr><th>Duration</th><td>{duration}</td></tr>
                    <tr><th>Frames Analyzed</th><td>{len(analyzed_frames)}</td></tr>
                    <tr><th>Resolution</th><td>{metadata['width']}x{metadata['height']}</td></tr>
                    <tr><th>Anomalies Detected</th><td>{len(anomalies)}</td></tr>
                </table>

                <h3>Analysis Summary</h3>
                <p>{summary}</p>
            </div>
        """

        if anomalies:
            html += """
            <div class="section">
                <h2>Detected Anomalies</h2>
            """

            for i, anomaly in enumerate(anomalies):
                html += f"""
                <div class="anomaly">
                    <h3>Anomaly #{i+1} - {anomaly['type'].capitalize()}</h3>
                    <table>
                        <tr><th>Timestamp</th><td>{anomaly['timestamp']}</td></tr>
                        <tr><th>Detection</th><td>{anomaly['caption']}</td></tr>
                        <tr><th>Keyword</th><td>{anomaly['keyword']}</td></tr>
                    </table>
                    <div class="frame">
                        <img src="{os.path.relpath(anomaly['path'], os.path.dirname(output_path))}" alt="Anomaly frame">
                    </div>
                </div>
                """

            html += "</div>"

        html += """
            <div class="section">
                <h2>Frame-by-Frame Analysis</h2>
                <div class="frames">
        """

        for frame in analyzed_frames:
            anomaly_class = "anomaly" if frame["anomaly"] else ""
            anomaly_text = f" - {frame['anomaly']['type'].capitalize()}" if frame["anomaly"] else ""

            html += f"""
            <div class="frame {anomaly_class}">
                <img src="{os.path.relpath(frame['path'], os.path.dirname(output_path))}" alt="Frame {frame['index']}">
                <p><strong>Time: {frame['formatted_time']}{anomaly_text}</strong></p>
                <p>{frame['caption']}</p>
            </div>
            """

        html += """
                </div>
            </div>
        </body>
        </html>
        """

        with open(output_path, 'w') as f:
            f.write(html)

    def generate_pdf_report(self, analyzed_frames, anomalies, summary, metadata, output_path):
        """Generate a PDF report of the analysis"""
        doc = SimpleDocTemplate(output_path, pagesize=letter)
        styles = getSampleStyleSheet()
        elements = []

        # Add title and metadata
        title_style = ParagraphStyle(
            'Title',
            parent=styles['Title'],
            fontSize=18,
            spaceAfter=12
        )
        elements.append(Paragraph("CCTV Footage Analysis Report", title_style))
        elements.append(Spacer(1, 0.2*inch))

        # Add timestamp
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        elements.append(Paragraph(f"Generated on: {timestamp}", styles["Normal"]))
        elements.append(Spacer(1, 0.2*inch))

        # Add video metadata
        duration = str(timedelta(seconds=int(metadata["duration"])))
        metadata_data = [
            ["Duration", duration],
            ["Frames Analyzed", str(len(analyzed_frames))],
            ["Resolution", f"{metadata['width']}x{metadata['height']}"],
            ["Anomalies Detected", str(len(anomalies))]
        ]

        metadata_table = Table(metadata_data, colWidths=[2*inch, 4*inch])
        metadata_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (0, -1), colors.lightgrey),
            ('TEXTCOLOR', (0, 0), (0, -1), colors.black),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
            ('FONTNAME', (1, 0), (1, -1), 'Helvetica'),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
            ('GRID', (0, 0), (-1, -1), 1, colors.black)
        ]))
        elements.append(metadata_table)
        elements.append(Spacer(1, 0.3*inch))

        # Add summary
        elements.append(Paragraph("Analysis Summary", styles["Heading2"]))
        elements.append(Paragraph(summary, styles["Normal"]))
        elements.append(Spacer(1, 0.3*inch))

        # Add anomalies section
        if anomalies:
            elements.append(Paragraph("Detected Anomalies", styles["Heading2"]))
            elements.append(Spacer(1, 0.1*inch))

            for i, anomaly in enumerate(anomalies):
                elements.append(Paragraph(f"Anomaly #{i+1}: {anomaly['type'].capitalize()}", styles["Heading3"]))

                anomaly_data = [
                    ["Timestamp", anomaly['timestamp']],
                    ["Detection", anomaly['caption']],
                    ["Keyword", anomaly['keyword']]
                ]

                anomaly_table = Table(anomaly_data, colWidths=[1.5*inch, 4.5*inch])
                anomaly_table.setStyle(TableStyle([
                    ('BACKGROUND', (0, 0), (0, -1), colors.pink),
                    ('TEXTCOLOR', (0, 0), (0, -1), colors.black),
                    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                    ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
                    ('FONTNAME', (1, 0), (1, -1), 'Helvetica'),
                    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
                    ('GRID', (0, 0), (-1, -1), 1, colors.black)
                ]))
                elements.append(anomaly_table)
                elements.append(Spacer(1, 0.1*inch))

                # Add anomaly image
                try:
                    img = RLImage(anomaly['path'], width=4*inch, height=3*inch)
                    elements.append(img)
                    elements.append(Spacer(1, 0.2*inch))
                except Exception as e:
                    elements.append(Paragraph(f"Error including image: {str(e)}", styles["Normal"]))

            elements.append(Spacer(1, 0.3*inch))

        # Add sample frames (2 frames from regular footage and 2 from anomalies if available)
        elements.append(Paragraph("Sample Frames", styles["Heading2"]))

        # Select sample frames - prioritize anomalies
        anomaly_frames = [frame for frame in analyzed_frames if frame["anomaly"]]
        normal_frames = [frame for frame in analyzed_frames if not frame["anomaly"]]

        sample_frames = []

        # Select up to 2 anomaly frames
        if anomaly_frames:
            # Space them out if possible
            if len(anomaly_frames) >= 2:
                sample_frames.append(anomaly_frames[0])
                sample_frames.append(anomaly_frames[len(anomaly_frames) // 2])
            else:
                sample_frames.extend(anomaly_frames)

        # Select up to 2 normal frames
        if normal_frames:
            # Space them out
            interval = max(1, len(normal_frames) // 2)
            indices = list(range(0, len(normal_frames), interval))[:2]
            for idx in indices:
                if idx < len(normal_frames):
                    sample_frames.append(normal_frames[idx])

        # Add sample frames to PDF
        for frame in sample_frames:
            caption = frame["caption"]
            frame_type = "Anomaly" if frame["anomaly"] else "Normal"
            elements.append(Paragraph(f"{frame_type} Frame - Time: {frame['formatted_time']}", styles["Heading4"]))
            elements.append(Paragraph(caption, styles["Normal"]))

            try:
                img = RLImage(frame['path'], width=4*inch, height=3*inch)
                elements.append(img)
                elements.append(Spacer(1, 0.2*inch))
            except Exception as e:
                elements.append(Paragraph(f"Error including image: {str(e)}", styles["Normal"]))

        doc.build(elements)

# Main execution code
def main():
    print("CCTV Anomaly Detection and Report Generation")
    print("===========================================")

    # Ask user to upload video or use from Google Drive
    use_drive = input("Do you want to use a video from Google Drive? (yes/no): ").lower() == "yes"

    if use_drive:
        # Use Google Drive path
        from google.colab import drive
        drive.mount('/content/drive')

        video_path = input("Enter the path to your video file in Google Drive (e.g., /content/drive/MyDrive/video.mp4): ")
        if not os.path.exists(video_path):
            print(f"Error: File not found at {video_path}")
            return
    else:
        # Upload video
        print("Please upload your CCTV footage file.")
        uploaded = files.upload()

        if not uploaded:
            print("No file was uploaded.")
            return

        video_path = list(uploaded.keys())[0]

    # Create output directory
    output_dir = "/content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results"
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    # Ask for frame interval
    try:
        frame_interval = int(input("Enter frame interval (higher value = faster processing, recommended 30-120): "))
    except ValueError:
        print("Invalid input, using default value of 60.")
        frame_interval = 60

    # Start analysis
    print("\nStarting analysis, this may take several minutes depending on video length...")
    analyzer = CCTVAnalyzer()

    start_time = time.time()
    results = analyzer.analyze_video(
        video_path=video_path,
        output_dir=output_dir,
        frame_interval=frame_interval
    )
    end_time = time.time()

    # Print results
    print(f"\nAnalysis completed in {end_time - start_time:.2f} seconds")
    print("\n===== ANALYSIS RESULTS =====")
    if results["anomalies"]:
        print(f"\nDetected {len(results['anomalies'])} potential anomalies:")
        for i, anomaly in enumerate(results["anomalies"]):
            print(f"• {anomaly['timestamp']} - {anomaly['caption']} (Type: {anomaly['type']})")
    else:
        print("\nNo anomalies detected in the footage")

    print(f"\nActivity summary: {results['summary']}")

    # Display HTML report in Colab
    from IPython.display import HTML, display
    with open(results["html_report"], 'r') as f:
        html_content = f.read()
    display(HTML(html_content))

    # Create ZIP file of results
    print("\nCreating downloadable ZIP file of all results...")
    !zip -r /content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results.zip {output_dir}

    # Generate download links
    print("\nDownload Links:")
    print("---------------")
    try:
        files.download('/content/drive/MyDrive/ReportGeneration/resultscctv_analysis_results.zip')
        print("✓ ZIP archive of all results (automatically downloading)")

        print(f"✓ PDF Report: {results['pdf_report']}")
        files.download(results['pdf_report'])

        print(f"✓ HTML Report: {results['html_report']}")
        files.download(results['html_report'])
    except Exception as e:
        print(f"Error creating download links: {str(e)}")
        print("You can manually download the files from the file browser.")

if __name__ == "__main__":
    main()

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
CCTV Anomaly Detection and Report Generation
Do you want to use a video from Google Drive? (yes/no): y
Please upload your CCTV footage file.


Saving Arson024_x264.mp4 to Arson024_x264.mp4
Enter frame interval (higher value = faster processing, recommended 30-120): 62

Starting analysis, this may take several minutes depending on video length...
Initializing models...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

Analyzing 55 frames...


  0%|          | 0/55 [00:00<?, ?it/s]


Analysis completed in 629.06 seconds

===== ANALYSIS RESULTS =====

Detected 44 potential anomalies:
• 0:00:00 - Arafed patient in a hospital room with a monitor and other medical equipment. (Type: emergency)
• 0:00:02 - Arafed hospital room with a man in a bed and a woman in a chair. (Type: crowd)
• 0:00:04 - Arafed patient in a hospital room with a laptop and other medical equipment. (Type: emergency)
• 0:00:06 - Arafed patient in a hospital room with a dog and other medical equipment. (Type: emergency)
• 0:00:08 - Arafed patient in a hospital room with a monitor and other medical equipment. (Type: emergency)
• 0:00:10 - Arafed patient in a hospital room with a monitor and other medical equipment. (Type: emergency)
• 0:00:12 - Arafed patient in a hospital room with a monitor and other medical equipment. (Type: emergency)
• 0:00:14 - Arafed patient in a hospital room with a monitor and other medical equipment. (Type: emergency)
• 0:00:16 - Arafed patient in a hospital room with a mon

Duration,0:01:52
Frames Analyzed,55
Resolution,320x240
Anomalies Detected,44
Timestamp,0:00:00
Detection,Arafed patient in a hospital room with a monitor and other medical equipment.
Keyword,medical
Timestamp,0:00:02
Detection,Arafed hospital room with a man in a bed and a woman in a chair.
Keyword,multiple people
Timestamp,0:00:04



Creating downloadable ZIP file of all results...
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/ (stored 0%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/ (stored 0%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/frame_0000.jpg (deflated 2%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/frame_0001.jpg (deflated 2%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/frame_0002.jpg (deflated 2%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/frame_0003.jpg (deflated 2%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/frame_0004.jpg (deflated 2%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysis_results/frames/frame_0005.jpg (deflated 2%)
  adding: content/drive/MyDrive/ReportGeneration/results/cctv_analysi